# fpl_v2 driver
Thin notebook over the package. All logic lives in the modules; this just runs and inspects.

Run jupyter from the repo root so `import fpl_v2` resolves (`uv run jupyter lab`).

In [1]:
# Ensure the repo root (the dir containing fpl_v2/) is importable, whatever the launch dir.
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / 'fpl_v2').is_dir() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [2]:
from fpl_v2 import pipeline, config

forecast, squad = pipeline.run(save=True)  # refresh=True to re-pull live data
forecast.shape

(425, 25)

In [3]:
cols = ['web_name', 'team_name', 'position', 'cost', 'xG', 'xAG', 'xClean', 'xPoints']
forecast.sort_values('xPoints', ascending=False)[cols].head(30)

,web_name,team_name,position,cost,xG,xAG,xClean,xPoints
294,Haaland,Man City,FWD,155,25.50,2.67,11.180432,110.017776
309,B.Fernandes,Man Utd,MID,120,10.79,12.28,10.119136,109.749319
373,Senesi,Spurs,DEF,60,1.53,4.72,8.799469,103.026483
0,Gabriel,Arsenal,DEF,80,2.94,1.75,15.890859,99.741872
241,Virgil,Liverpool,DEF,65,3.77,1.44,9.206913,97.756410
180,Tarkowski,Everton,DEF,60,2.53,2.10,7.837359,93.606286
371,Van Hecke,Spurs,DEF,50,3.31,1.57,8.799469,92.434733
271,Guéhi,Man City,DEF,60,4.05,2.37,11.180432,91.942866
140,Enzo,Chelsea,MID,70,11.26,7.26,8.232687,89.432913
94,Thiago,Brentford,FWD,80,20.60,1.83,8.020974,89.150713


In [4]:
squad_cols = ['web_name', 'team_name', 'position', 'cost', 'xPoints']
print(f"{squad.formation}  |  £{squad.total_cost/10:.1f}m  |  total xP (capt x2): {squad.total_xpoints:.1f}")
print('Captain:', squad.captain['web_name'])
squad.players.sort_values(['position', 'xPoints'], ascending=[True, False])[squad_cols]

5-4-1  |  £82.5m  |  total xP (capt x2): 1127.0
Captain: Haaland


,web_name,team_name,position,cost,xPoints
373,Senesi,Spurs,DEF,60,103.026483
241,Virgil,Liverpool,DEF,65,97.756410
180,Tarkowski,Everton,DEF,60,93.606286
371,Van Hecke,Spurs,DEF,50,92.434733
271,Guéhi,Man City,DEF,60,91.942866
294,Haaland,Man City,FWD,155,110.017776
425,Raya,Arsenal,GK,60,65.741636
309,B.Fernandes,Man Utd,MID,120,109.749319
140,Enzo,Chelsea,MID,70,89.432913
295,Anderson,Man City,MID,65,84.915728


## Tweaking
- Scoring weights: `config.POINTS_FOR_GOAL`, `POINTS_FOR_ASSIST`, `POINTS_FOR_CLEAN`
- Formations / budgets: `config.FORMATIONS`
- Multi-season blend: `config.BLEND_WEIGHTS`
- Manual fixes: `overrides.OVERRIDES`

e.g. re-run a single formation:

## Goalkeepers
GK xPoints ranking (the keeper is also folded into the squad optimiser above).

In [5]:
from fpl_v2 import goalkeepers

goalkeepers.rank().head(10)

,web_name,team_name,now_cost,s90,saves_per_90,xga_pg,xClean,GK_xPoints
0,Raya,Arsenal,60,37.0,1.62,0.871842,15.890859,65.741636
1,Dubravka,Spurs,40,35.0,3.63,1.462895,8.799469,49.168440
2,Verbruggen,Brighton,45,38.0,2.79,1.414474,9.236033,45.409132
3,Donnarumma,Man City,55,34.0,2.29,1.223421,11.180432,45.169353
4,Petrović,Bournemouth,45,38.0,2.87,1.493947,8.530422,42.090020
5,Lammens,Man Utd,50,32.0,2.47,1.323158,10.119136,39.261650
6,Kelleher,Brentford,50,37.0,2.95,1.555526,8.020974,38.845679
7,Roefs,Sunderland,50,35.0,3.11,1.587895,7.765504,37.104929
8,Martinez,Aston Villa,50,31.5,3.02,1.491579,8.550649,36.569784
9,Henderson,Crystal Palace,50,37.0,2.86,1.580263,7.824994,36.514758


## Defensive contribution & transfers
Expected DefCon points (in `xPoints`), plus movers flagged for manual review.

In [6]:
dc = ['web_name', 'team_name', 'position', 'expected_defcon_points', 'is_mover', 'defcon_multiplier']
print('Top defensive-contribution scorers:')
print(forecast.sort_values('expected_defcon_points', ascending=False)[dc].head(8).to_string(index=False))
print('\nTransferred players flagged for manual review (multiplier < 1 = moved to a less defensive side):')
forecast[forecast['is_mover']].sort_values('defcon_multiplier')[dc]

Top defensive-contribution scorers:
 web_name      team_name position  expected_defcon_points  is_mover  defcon_multiplier
   Senesi          Spurs      DEF               45.847120      True           0.934024
 Anderson       Man City      MID               44.952980      True           0.907172
  Lacroix Crystal Palace      DEF               43.652061     False           1.000000
Tarkowski        Everton      DEF               41.601834     False           1.000000
   Garner        Everton      MID               41.503601     False           1.000000
   Ampadu          Leeds      MID               37.347780     False           1.000000
 Andersen         Fulham      DEF               36.880747     False           1.000000
    Scott    Bournemouth      MID               35.495854     False           1.000000

Transferred players flagged for manual review (multiplier < 1 = moved to a less defensive side):


,web_name,team_name,position,expected_defcon_points,is_mover,defcon_multiplier
47,Gomes,Aston Villa,MID,16.359903,True,0.833647
36,Guessand,Aston Villa,MID,1.073512,True,0.851253
20,Nelson,Arsenal,MID,2.174205,True,0.867376
269,Grealish,Man City,MID,1.329577,True,0.881830
295,Anderson,Man City,MID,44.952980,True,0.907172
138,Disasi,Chelsea,DEF,0.000000,True,0.907911
50,Garnacho,Aston Villa,MID,0.094095,True,0.926388
373,Senesi,Spurs,DEF,45.847120,True,0.934024
112,Buonanotte,Brighton,MID,0.052465,True,0.952293
124,Struijk,Brighton,DEF,24.104389,True,0.952293


In [7]:
from fpl_v2 import optimize

# Optimise over the full pool (GK + outfield) for a single chosen formation.
pool = pipeline.player_pool()
alt = optimize.optimize(pool, formations={'3-4-3': config.FORMATIONS['3-4-3']})
alt.players.sort_values('xPoints', ascending=False)[squad_cols]

,web_name,team_name,position,cost,xPoints
294,Haaland,Man City,FWD,155,110.017776
309,B.Fernandes,Man Utd,MID,120,109.749319
373,Senesi,Spurs,DEF,60,103.026483
241,Virgil,Liverpool,DEF,65,97.756410
371,Van Hecke,Spurs,DEF,50,92.434733
140,Enzo,Chelsea,MID,70,89.432913
94,Thiago,Brentford,FWD,80,89.150713
295,Anderson,Man City,MID,65,84.915728
189,Garner,Everton,MID,60,78.364919
237,Calvert-Lewin,Leeds,FWD,60,65.476315
